# Final OCSD on the CLEAN farm sketch (module slides M1-M5)

Same caption as before ("three small cows, a red truck and a tall tree on a farm", 5 objects), but the sketch is the clean layout from slide 4: no two objects touch, so no mask overlaps (the benchmark scene qd_5_medium_005 had a cow drawn over the truck). The scene folder is in the repo: `notebooks/assets/scenes/farm_clean` (built by `tools/make_farm_clean_scene.py` with the same rendering, masks and relations as the benchmark).
Runs the **final OCSD** (SD 1.5, M1-M5, M3 on, harmonised M4 from `results/m4_study_v1/locked.json`) for **seeds 0 and 1**; for each seed it saves (a) every module's intermediate images and (b) the same sketch with 3 other backgrounds.

**How to run:** Runtime -> Change runtime type -> any GPU (A100 fastest; T4 or L4 also work) -> **Run all**. About 40-60 minutes on A100. One ZIP downloads at the end; it is also on Drive at `KLTN_OCSD/trace_ocsd/EXAMPLE_farm_clean.zip`.


In [ ]:
#@title 1. Settings
SID    = "farm_clean"
SEEDS  = [0, 1]
BGS    = ["in a city park", "on a snowy field", "on a sandy beach"]
BRANCH = "claude/project-thread-srtlwl"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"  #@param {type:"string"}


In [ ]:
#@title 2. Drive, GPU check, code, libraries
import os, subprocess, sys, shutil
from google.colab import drive
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode or int(gpu.stdout.strip().split(",")[-1]) < 14000:
    raise RuntimeError("This needs a GPU with at least 14 GB (T4, L4 or A100): Runtime > Change runtime type > GPU, then Run all")
# OCSD on SD 1.5 (with M3 training) fits in about 12 GB, so a T4, L4 or A100 all work
CODE = "/content/KLTN_OCSD_example"
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/bobbibao/ocsd-thesis.git", CODE], check=True)
print("Code:", subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
os.environ["HF_HOME"] = "/content/hf_cache"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
print(r.stdout[-1000:], r.stderr[-2000:])
if r.returncode:
    raise RuntimeError("pip install failed (see the text above)")
from ocsd import hfcache
hfcache.repair(os.environ["HF_HOME"])

In [ ]:
#@title 3. Final OCSD on the clean sketch: steps, M2 scores, attention + own background + 3 others, then the module tracer (each seed)
from ocsd import bg_swap, trace_ocsd
import gc, torch
SCENE = os.path.join(CODE, "notebooks", "assets", "scenes", SID)
FINAL = bg_swap.final_overrides(CODE)
print("final OCSD settings:", FINAL)
TOUT = os.path.join(DRIVE_ROOT, "trace_ocsd", "final")
for SEED in SEEDS:
    META = bg_swap.run(DRIVE_ROOT, sid=SID, seed=SEED, bgs=BGS, scene_dir=SCENE, overrides=FINAL, tag="_final",
                       record_steps=(0, 5, 11, 17, 23, 29), attn_steps=(11, 23))
    for r in META["runs"]: print(SEED, r["bg"], "|", r["global_prompt"], "|", r["seconds"], "s, tries", r["log"].get("tries"))
    gc.collect(); torch.cuda.empty_cache()
    TMETA = trace_ocsd.trace(DRIVE_ROOT, sid=SID, seed=SEED, scene_dir=SCENE, overrides=FINAL, out_root=TOUT)
    print(SEED, "M3 check rows:", len(TMETA["m3"]["check"]), "| M5(d) tries:", TMETA["m5d"]["tries"])
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
#@title 4. One ZIP with everything, then download
import zipfile
from IPython.display import Image, display
zpath = os.path.join(DRIVE_ROOT, "trace_ocsd", f"EXAMPLE_{SID}.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for SEED in SEEDS:
        b_out = os.path.join(DRIVE_ROOT, "trace_ocsd", f"bgswap_{SID}_s{SEED}_final")
        t_out = os.path.join(TOUT, f"{SID}_s{SEED}")
        display(Image(os.path.join(b_out, "SHEET.jpg"), width=1400))
        for root, name in ((b_out, f"bgswap_s{SEED}"), (t_out, f"trace_s{SEED}")):
            for dp, _, fs in os.walk(root):
                for f in fs:
                    p = os.path.join(dp, f)
                    z.write(p, os.path.join(name, os.path.relpath(p, root)))
print("ZIP:", zpath, os.path.getsize(zpath) // 1024, "KB")
from google.colab import files
try:
    files.download(zpath)
except Exception as exc:
    print("Get it from Drive:", zpath, exc)
print("Done. Disconnect the GPU runtime.")
